# DISCOtoolkit quickstart (DISCO v1)

Find samples in the [DISCO](https://disco.bii.a-star.edu.sg/) database by metadata and cell type, download them, and search a gene's expression.
This notebook runs in Google Colab or any Python >= 3.9. It takes a few minutes the first time (the install pulls in scanpy).

## 1. Install

In [ ]:
!pip install -q -U discotoolkit

# or the latest development version from GitHub:
# !pip install -q "git+https://github.com/JinmiaoChenLab/DISCOtoolkit_py.git"

## 2. Check the install and the server

The toolkit talks to DISCO v1 by default. `dt.set_server("v2")` or `dt.set_server("https://.../disco_v3_api/")` switches server.

In [ ]:
import discotoolkit as dt

print("discotoolkit", dt.__version__)
print("server:", dt.get_server())

## 3. (Optional) Check that the server answers everything the toolkit needs

This is the same check the maintainers run. Every line should say PASS.

In [ ]:
!git clone -q --depth 1 https://github.com/JinmiaoChenLab/DISCOtoolkit_py.git /content/DISCOtoolkit_py
!python /content/DISCOtoolkit_py/tests/test_server_contract.py

## 4. Explore what is in DISCO

In [ ]:
print(sorted(map(str, dt.list_metadata_item("tissue")))[:15])      # tissues
print(dt.find_celltype("myoid"))                                    # search the cell ontology

## 5. Filter samples

Filter by tissue, disease, platform, project, sample type or cell type. A cell type filter returns the samples that *contain* that cell type, and the download keeps only those cells -- handy for rare ones, such as **thymic myoid cells**.

In [ ]:
flt = dt.Filter(
    tissue="thymus",
    platform=["10x3'"],
    sample_type=["control"],
    cell_type="Thymic myoid cell",
    cell_type_confidence="medium",     # "high", "medium" or "all"
    min_cell_per_sample=100,
)
metadata = dt.filter_disco_metadata(flt)
print(metadata.sample_count, "samples,", metadata.cell_count, "cells")
metadata.sample_metadata.head()

## 6. Download

Each sample is a 10x `.h5` converted to an AnnData `.h5ad`, with DISCO's cell type annotation in `obs["cell_type"]`. To have something to look at, take three small control-thymus samples from one study (all their cells, about 8,000).

In [ ]:
flt = dt.Filter(sample_id=["ERX3806129", "ERX3806130", "ERX3806131"])
metadata = dt.filter_disco_metadata(flt)
print(metadata.sample_count, "samples,", metadata.cell_count, "cells")

dt.download_disco_data(metadata, output_dir="disco_data")   # one .h5ad per sample
!ls -lh disco_data

## 7. A UMAP of the downloaded cells

The files are standard AnnData, so scanpy works directly. Coloured by DISCO's labels the UMAP shows the cell types; coloured by sample it shows that the three samples mix.

In [ ]:
import anndata as ad
import scanpy as sc

ids = list(metadata.sample_metadata["sample_id"])
adata = ad.concat(
    [sc.read_h5ad(f"disco_data/{s}.h5ad") for s in ids],
    keys=ids, index_unique="_",          # barcodes repeat between samples
)
print(adata.obs["cell_type"].value_counts().head(5))

# the standard scanpy workflow: normalise, pick variable genes, embed
sc.pp.filter_genes(adata, min_cells=10)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=2000)
sc.pp.pca(adata, mask_var="highly_variable")
sc.pp.neighbors(adata)
sc.tl.umap(adata)

sc.pl.umap(adata, color=["cell_type", "sample_id"], wspace=0.5)

## 8. Marker genes

Expression should agree with the labels: `MKI67` marks the cycling thymocytes and `PTCRA` the double-positive stage.

In [ ]:
sc.pl.umap(adata, color=["CD4", "CD8A", "PTCRA", "MKI67"], ncols=2, cmap="viridis")

## 9. Cells per type

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

counts = pd.crosstab(adata.obs["cell_type"], adata.obs["sample_id"])
top = counts.loc[counts.sum(axis=1).nlargest(8).index]
top.plot.barh(stacked=True, figsize=(6, 3.5))
plt.xlabel("cells"); plt.tight_layout(); plt.show()

## 10. Search a gene across DISCO

Without downloading anything: a gene's expression across all annotated cell types, as on the DISCO website. ACTA1 is highest in skeletal muscle, and also in thymic myoid cells.

In [ ]:
dt.gene_search("ACTA1", atlas=["skeletal_muscle", "thymus"])

## More

- Cell type annotation with CELLiD: [CELLiD_celltype_annotation.ipynb](CELLiD_celltype_annotation.ipynb)
- Gene set enrichment: [scEnrichment.ipynb](scEnrichment.ipynb)
- Filtering and downloading in depth: [download_data.ipynb](download_data.ipynb)